In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re

def top_killers(n, sort_by="confirmed-kills", location="world"):
    
    if " " in location:
        location = location.replace(" ","-")
        
    BASE_URL = "https://killer.cloud"
    HEADERS = {"User-Agent": "Mozilla/5.0"}
    sort_by = sort_by.replace(" ", "-")
    killers = []
    page = 1

    while len(killers) < n:
        time.sleep(1)

        if location == "world":
            url = f"{BASE_URL}/serial-killers/by/{sort_by}?page={page}"
        else:
            url = f"{BASE_URL}/serial-killers/by/country/{location.lower()}?page={page}"

        response = requests.get(url, headers=HEADERS)
        soup = BeautifulSoup(response.content, "html.parser")

        # same structure for both world and country
        links = soup.select('a[href*="/serial-killers/show/"]')

        if not links:
            break

        for a in links:
            try:
                name    = a.find('h3', class_=re.compile(r'pad10t mar5t font-hel text-left')).get_text()
                victims = int(a.find('h4', class_=re.compile(r'mar0 pad10t \w font-hel')).get_text())
                years   = int(a.find('h4', class_=re.compile(r'mar0 pad15tb \w font-hel')).get_text())
                url     = a['href']

                killers.append({"name": name, "victims": victims, "years": years, "url": url})

                if len(killers) >= n:
                    break
            except:
                continue

        page += 1

    return killers

location = input("Enter country (or 'world'): ")
sort_by  = input("Sort by (confirmed-kills / years-active): ")
n        = int(input("How many killers? "))
print(top_killers(n, sort_by=sort_by, location=location))

In [ ]:
import requests
from bs4 import BeautifulSoup
import re


def profile(name):
   
    search_url = f"https://killer.cloud/search?term={name.replace(' ', '+')}"
    
    HEADERS = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(search_url, headers=HEADERS)
    soup = BeautifulSoup(response.content, "html.parser")

    
    link = soup.find("a", href=re.compile(r"/serial-killers/show/"))
    if not link:
        return {}

    profile_url = link["href"]

    response = requests.get(profile_url, headers=HEADERS)
    soup = BeautifulSoup(response.content, "html.parser")

    data = {"name": name}

    for row in soup.select("table tr"):
        th = row.find("th")
        td = row.find("td")
        if not th or not td:
            continue

        key = th.get_text(strip=True).rstrip(":").lower().replace(" ", "_").replace("-", "_")

        if key == "nationality":
            key = "citizenship"

        value = td.get_text(strip=True)
        data[key] = value

    return {
        "name":         data.get("name"),
        "nickname":     data.get("nickname"),
        "victims":      data.get("victims"),
        "years_active": data.get("years_active"),
        "countries":    data.get("active_countries"),
        "gender":       data.get("gender"),
        "nationality":  data.get("citizenship"),
        "iq":           data.get("iq"),
        "arrested":     data.get("arrested"),
        "convicted":    data.get("convicted"),
        "sentence":     data.get("sentence"),
    }

name = input("Enter name: ")
print(profile(name))

In [ ]:
import requests
from bs4 import BeautifulSoup
import unicodedata

def normalize(text):
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()

def profilePicture(name):
    base = "https://murderpedia.org"
    HEADERS = {"User-Agent": "Mozilla/5.0"}

    # normalize name (fix accents like András → Andras)
    name = normalize(name)

    parts = name.lower().split()
    firstname = parts[0]
    lastname = parts[-1]
    letter = lastname[0].upper()

    categories = ["male", "female"]  # just try both cleanly

    for category in categories:
        for i in range(5):
            suffix = f"{letter}{i}" if i > 0 else letter

            url = f"{base}/{category}.{suffix}/{suffix.lower()}/{lastname}-{firstname}.htm"
            print(url)

            response = requests.get(url, headers=HEADERS)
            if response.status_code != 200:
                continue

            soup = BeautifulSoup(response.content, "html.parser")

            # find main image
            img = soup.find("img", src=True)
            if not img:
                continue

            src = img["src"]

            # must be relative image path
            if not src.startswith("../images"):
                continue

            # build full image URL
            page_dir = response.url.rsplit("/", 1)[0]
            parent_dir = page_dir.rsplit("/", 1)[0]
            img_url = parent_dir + "/" + src.replace("../", "")

            # download image
            img_data = requests.get(img_url, headers=HEADERS).content

            filename = f"{name.lower().replace(' ', '_')}.jpg"
            with open(filename, "wb") as f:
                f.write(img_data)

            return filename

    return "No image found."


# run
name = input("Enter killer name: ")
print(profilePicture(name))

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import json
import os
import unicodedata

def profilePicture(name):

    naam = name
    name = unicodedata.normalize('NFKD', name)
    name = ''.join(c for c in name if not unicodedata.combining(c))

    name = name.split(" ")
    lastName = name[-1]
    initial = name[-1][0]
    nameLink = name[-1] + "-" + "-".join(name[:-1])
    
    
    sex = "Male"
    url = rf"https://murderpedia.org/{sex.lower()}.{initial}/{initial.lower()}/{nameLink.lower()}.htm"
    headers = {'User-Agent' : 'Anthony'}
    response = requests.get(url, headers=headers)
    code = response.status_code
    
    num = 1
    if code != 200:
        while code != 200:
                url = rf"https://murderpedia.org/{sex.lower()}.{initial}/{initial.lower()}{num}/{nameLink.lower()}.htm"
                num += 1 
                headers = {'User-Agent' : 'Anthony'}
                response = requests.get(url, headers=headers)
                code = response.status_code

                if num == 5:
                    break
    if code != 200:
        sex = "Female"
        url = rf"https://murderpedia.org/{sex.lower()}.{initial}/{initial.lower()}/{nameLink.lower()}.htm"
        print(url)
        headers = {'User-Agent' : 'Anthony'}
        response = requests.get(url, headers=headers)
        code = response.status_code
        
        if code != 200:
            while code != 200:
                url = rf"https://murderpedia.org/{sex.lower()}.{initial}/{initial.lower()}{num}/{nameLink.lower()}.htm"
                num += 1 
                headers = {'User-Agent' : 'Anthony'}
                response = requests.get(url, headers=headers)
                code = response.status_code

                if num == 5:
                    break
    
    soup = BeautifulSoup(response.content, 'html.parser')
    image = soup.find_all("img")
    image_url = ""
    for img in image:
        
        imageLink = img["src"]
        if imageLink.startswith("../images"):
            image_url = imageLink
            break

    part = f"https://murderpedia.org/{sex.lower()}.{initial}"
    
    image_url = image_url.replace("..", part)

    filename = f"""{naam.replace(" ","_").lower()}""" + ".jpg"
    
    r = requests.get(image_url, allow_redirects=True)
    with open(filename, 'wb') as file:
        file.write(r.content)
    return filename

name = input()
profilePicture(name)

'marc_dutroux.jpg'

In [ ]:
import requests
from bs4 import BeautifulSoup

HEADERS = {'User-Agent': 'Mozilla/5.0'}
BASE = "https://murderpedia.org"


def build_url(sex, initial, name_link, suffix=""):
    return f"{BASE}/{sex}.{initial}/{initial.lower()}{suffix}/{name_link}.htm"


def get_valid_page(initial, name_link):
    for sex in ["male", "female"]:
        url = build_url(sex, initial, name_link)
        res = requests.get(url, headers=HEADERS)

        if res.status_code == 200:
            return res, sex, initial
   
        for i in range(1, 5):
            url = build_url(sex, initial, name_link, i)
            res = requests.get(url, headers=HEADERS)

            if res.status_code == 200:
                return res, sex, initial
    return None, None, None


def extract_image(response, sex, initial):
    soup = BeautifulSoup(response.content, "html.parser")
    for img in soup.find_all("img"):
        src = img.get("src", "")
        if src.startswith("../images"):
            base_path = f"{BASE}/{sex}.{initial}"
            return src.replace("..", base_path)

    return None


def profilePicture(name):
    original_name = name

    parts = name.lower().split()
    last = parts[-1]
    initial = last[0].upper()
    name_link = f"{last}-{'-'.join(parts[:-1])}"

    response, sex, initial = get_valid_page(initial, name_link)

    if not response:
        return "No image found."

    img_url = extract_image(response, sex, initial)

    if not img_url:
        return "No image found."

    filename = f"{original_name.replace(' ', '_').lower()}.jpg"

    img_data = requests.get(img_url, headers=HEADERS).content
    with open(filename, "wb") as f:
        f.write(img_data)

    return filename



name = input("Enter name: ")
print(profilePicture(name))

edmund_kemper.jpg


In [ ]:
import requests
from bs4 import BeautifulSoup

def profilePicture(name):
    BASE = "https://murderpedia.org"
    HEADERS = {'User-Agent': 'Mozilla/5.0'}

    original_name = name
    parts = name.lower().split()

    last = parts[-1]
    first_part = "-".join(parts[:-1])
    name_link = f"{last}-{first_part}"
    initial = last[0].upper()


    for sex in ["male", "female"]:
        url = f"{BASE}/{sex}.{initial}/{initial.lower()}/{name_link}.htm"
        res = requests.get(url, headers=HEADERS)

        if res.status_code != 200:
        
            for i in range(1, 5):
                url = f"{BASE}/{sex}.{initial}/{initial.lower()}{i}/{name_link}.htm"
                res = requests.get(url, headers=HEADERS)
                if res.status_code == 200:
                    break
            else:
                continue 

        soup = BeautifulSoup(res.content, "html.parser")

        img_url = None
        for img in soup.find_all("img"):
            src = img.get("src", "")
            if src.startswith("../images"):
                base_path = f"{BASE}/{sex}.{initial}"
                img_url = src.replace("..", base_path)
                break

        if not img_url:
            continue

     
        filename = f"{original_name.replace(' ', '_').lower()}.jpg"
        img_data = requests.get(img_url, headers=HEADERS).content

        with open(filename, "wb") as f:
            f.write(img_data)

        return filename

    return "No image found."


# run
name = input("Enter name: ")
print(profilePicture(name))

edmund_kemper.jpg
